<a href="https://colab.research.google.com/github/berkemremert/eva_dialog_trial_collab/blob/main/eva_dialog_trial_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ultravox v0.7 — Colab setup

The notebook is split so expensive work is not repeated:

1. Run **Environment**, **Configuration**, and **Hugging Face login** once per Colab session.
2. Leave cache cleanup disabled unless remote model code is broken or stale.
3. Run **Load model** once. Re-running that cell skips loading while `model` is already in memory.
4. Run the lightweight memory/status cell whenever needed.

> The first model download is still large and may take a long time. Colab/Hugging Face cache makes later sessions faster as long as the runtime cache survives.

## 1. Environment and imports

Run once after connecting to a GPU runtime.

In [ ]:
import gc
import os
import shutil

import huggingface_hub
import torch
import transformers
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModel

print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(properties.total_memory / 1024**3, 1), "GB")
else:
    print("⚠️ In Colab, select Runtime → Change runtime type → GPU.")

## 2. Configuration

Keep both switches `False` for normal use.

In [ ]:
MODEL_ID = "fixie-ai/ultravox-v0_7-glm-4_6"
CLEAR_REMOTE_CODE_CACHE = False
FORCE_RELOAD = False

print("Model:", MODEL_ID)
print("Clear remote-code cache:", CLEAR_REMOTE_CODE_CACHE)
print("Force model reload:", FORCE_RELOAD)

## 3. Hugging Face login

Add `HF_TOKEN` under Colab's **Secrets** panel before running this cell.

In [ ]:
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Add HF_TOKEN to Colab Secrets and allow this notebook to access it."
    ) from exc

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty. Update it in Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Hugging Face login successful")

## 4. Optional remote-code cache cleanup

This is normally skipped. Enable `CLEAR_REMOTE_CODE_CACHE` in Configuration only when a stale remote-code error needs fixing. It does not delete the downloaded model weights.

In [ ]:
REMOTE_CODE_CACHE = os.path.expanduser(
    "~/.cache/huggingface/modules/transformers_modules"
)

if CLEAR_REMOTE_CODE_CACHE:
    if os.path.exists(REMOTE_CODE_CACHE):
        shutil.rmtree(REMOTE_CODE_CACHE)
        print("✅ Transformers remote-code cache cleared")
    else:
        print("Remote-code cache is already empty")
else:
    print("⏭️ Cache cleanup skipped")

## 5. Load Ultravox v0.7 (slow — run once)

This is the expensive block. If the model is already loaded, re-running the cell skips it. Set `FORCE_RELOAD = True` only when you intentionally want to discard and reload the in-memory model.

In [ ]:
model_is_loaded = "model" in globals() and model is not None

if model_is_loaded and not FORCE_RELOAD:
    print("✅ Model is already loaded; skipping reload")
else:
    if model_is_loaded:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Loading:", MODEL_ID)
    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="auto",
        dtype=torch.float16,
        token=HF_TOKEN,
    )
    model.eval()
    print("✅ Ultravox v0.7 GLM loaded successfully")

## 6. Model and GPU status

This block is quick and safe to rerun.

In [ ]:
print("Model loaded:", "model" in globals() and model is not None)

if torch.cuda.is_available():
    print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
    print("GPU reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")